In [49]:
using Lux, DiffEqFlux, OrdinaryDiffEq, Plots, Printf, Statistics
using ComponentArrays
using Optimization, OptimizationOptimisers,OptimizationOptimJL
#using Optimisers
using Enzyme
using Dates
using Random
using StaticArrays
using SciMLSensitivity
using SciMLStructures

Enzyme.API.looseTypeAnalysis!(true)

In [50]:
function evolve!(dc, c, p, t)
    e1 = p.e
    e2 = 10
    dc .= c * p.p2 * p.p1 .+ e1 .+ e2 .+ d
end

function simulate(c0, i1, i2, a, b, para_sim, t_span)
    c,d,e = para_sim
    p2 = exp(-i2 * a)
    p1 = i1 * b
    p = (p1, p2, c, d, e)
    p_named = NamedTuple{(:p1, :p2, :c, :d, :e)}(p)
    #p_named = NamedTuple{(:p1, :p2)}(p)

    p = ComponentArray(p_named)
    #display(p)
    #c0 = [1.0 2.0; 1.0 0.0]
    prob = ODEProblem(evolve!, c0, t_span, p)
    sol = solve(prob, Euler(), dt=0.5)
    return Array(sol[end])
end

simulate (generic function with 1 method)

In [51]:
rng = Xoshiro(0)
b = [0.0 1.0; 1.0 0.0]
a = 0.6
c = 1.
d = 2.
e = 3
para_sim = c, d, e
n = length(b[1, :])
println("n:", n)
println("b:", b)
i1 = 0.18
i2 = 2.5 
timespan = (0.0, 5.0)
"""
p2 = exp(-i2 * a)
p1 = i1 * b
p = (p1, p2, c, d, e)
p_named = NamedTuple{(:p1, :p2, :c, :d, :e)}(p)
p = ComponentArray(p_named)
"""
c0 = [1.0 2.0; 1.0 0.0]
sol = simulate(c0, i1, i2, a, b, para_sim, timespan)
#prob = ODEProblem(evolve!, c0, timespan, p)
#sol = solve(prob, Euler(), save_everystep = false, dt = 0.5)
#sol = solve(prob, Tsit5())
ans = Array(sol)

n:2
b:[0.0 1.0; 1.0 0.0]


2×2 Matrix{Float64}:
 83.5754  84.3917
 83.1718  82.3554

In [52]:
inputs = [i1, i2]
input_size = length(inputs)
output_size = length(a) + length(b)
nn = Chain(
    Dense(input_size, input_size*3*n, tanh),
    Dense(input_size*3*n, output_size*2, tanh),
    Dense(output_size*2, output_size, sigmoid)
)

u, st = Lux.setup(rng, nn)

((layer_1 = (weight = Float32[-1.8019577 -1.1146251; -0.18273845 1.0075601; … ; 1.6416063 0.04306274; 1.0081401 -1.5636357], bias = Float32[-0.022002257, 0.5653541, 0.48737866, -0.109010376, 0.25698113, 0.29349717, -0.0496704, 0.5392088, 0.5833104, 0.61079556, -0.6089294, 0.47623715]), layer_2 = (weight = Float32[0.20824511 0.08736193 … 0.29591995 -0.76305753; -0.17639339 0.37737656 … -0.5897239 -0.32251218; … ; 0.52063733 -0.4213308 … -0.34941265 0.34940132; -0.40962756 0.7908495 … 0.2728175 -0.6145144], bias = Float32[-0.030394312, -0.11422878, 0.14924575, 0.082523964, -0.056747876, -0.24036361, -0.16483621, -0.10791249, -0.19515306, -0.023660526]), layer_3 = (weight = Float32[0.4909926 0.39306656 … -0.28983527 -0.15333983; -0.11188733 -0.5009294 … -0.2942955 0.279714; … ; 0.19163486 -0.22708948 … -0.20597327 0.05595271; -0.18737693 0.18280187 … 0.4284568 0.2179301], bias = Float32[-0.27912286, -0.2785868, 0.062313087, 0.078882255, -0.16884351])), (layer_1 = NamedTuple(), layer_2 = N

In [53]:
function predict_neuralode(u)
    # Get parameters from the neural network
    output, outst = nn(inputs, u, st)

    # Segregate the output
    p_a = output[1]
    pp_b = output[length(a)+1:end]
    p_b = zeros(n, n)
    index = 1
    for i in 1:n
        for j in 1:n
            p_b[i, j] = pp_b[index]
            index += 1
        end
    end

    nn_output = [p_a, p_b]
    pred = simulate(c0, i1, i2, p_a, p_b, para_sim, timespan)
    return Array(pred)
end

function loss_neuralode(u)
    pred = predict_neuralode(u)
    loss = sum(abs2, ans .- pred)
    return loss, pred
end

loss_neuralode (generic function with 1 method)

In [54]:
loss_values = []
predictions = []

callback = function (state::Optimization.OptimizationState, loss_value::Float64; doplot = false)
    p = state.u
    l, pred = loss_neuralode(p)
    println(l)
    #push!(loss_values[1], l)
    # plot current prediction against data
    push!(loss_values, l)
    push!(predictions, pred)
    if doplot
        plt = scatter(tsteps, ans[1,1], label = "Phase 1 Data", color = :blue)
        scatter!(plt, tsteps, ans[1,2], label = "Phase 2 Data", color = :red)
        scatter!(plt, tsteps, ans[2,1], label = "Phase 3 Data", color = :green)
        scatter!(plt, tsteps, ans[2,2], label = "Phase 4 Data", color = :yellow)
        scatter!(plt, tsteps, pred[1,1], label = "Phase 1 Prediction", color = :blue, shape = :cross)
        scatter!(plt, tsteps, pred[1,2], label = "Phase 2 Prediction", color = :red, shape = :cross)
        scatter!(plt, tsteps, pred[2,1], label = "Phase 3 Prediction", color = :green, shape = :cross)
        scatter!(plt, tsteps, pred[2,2], label = "Phase 4 Prediction", color = :yellow)
        display(plot(plt))
        savefig(plt, "training_$timestamp.svg")
    end
    return false
end

#46 (generic function with 1 method)

In [55]:
pinit = ComponentArray(u)
#callback(pinit, loss_neuralode(pinit)[1])

adtype = Optimization.AutoEnzyme(; mode=set_runtime_activity(Reverse))

optf = Optimization.OptimizationFunction((x,_) -> loss_neuralode(x), adtype)
optprob = Optimization.OptimizationProblem(optf, pinit)

result_neuralode = Optimization.solve(
    optprob, OptimizationOptimisers.Adam(0.02); callback = callback, maxiters = 5)

720.3524647600619
8045.33912813975
115116.6062716953
1.6602814730716972e6
2.0855931575717144e7
2.253520324333289e8


retcode: Default
u: ComponentVector{Float32}(layer_1 = (weight = Float32[-1.8019577 -1.1146251; -0.18273845 1.0075601; … ; 1.6416063 0.04306274; 1.0081401 -1.5636357], bias = Float32[-0.022002257, 0.5653541, 0.48737866, -0.109010376, 0.25698113, 0.29349717, -0.0496704, 0.5392088, 0.5833104, 0.61079556, -0.6089294, 0.47623715]), layer_2 = (weight = Float32[0.20824511 0.08736193 … 0.29591995 -0.76305753; -0.17639339 0.37737656 … -0.5897239 -0.32251218; … ; 0.52063733 -0.4213308 … -0.34941265 0.34940132; -0.40962756 0.7908495 … 0.2728175 -0.6145144], bias = Float32[-0.030394312, -0.11422878, 0.14924575, 0.082523964, -0.056747876, -0.24036361, -0.16483621, -0.10791249, -0.19515306, -0.023660526]), layer_3 = (weight = Float32[0.4909926 0.39306656 … -0.28983527 -0.15333983; -0.11188733 -0.5009294 … -0.2942955 0.279714; … ; 0.19163486 -0.22708948 … -0.20597327 0.05595271; -0.18737693 0.18280187 … 0.4284568 0.2179301], bias = Float32[-0.27912286, -0.2785868, 0.062313087, 0.078882255, -0.168843